# Leather Defect Segmentation v5 — Attention UNet + ResNet50

**Architecture:** Attention UNet with pre-trained ResNet50 encoder + Squeeze-Excite + Deep Supervision  
**Loss:** Weighted Focal Cross-Entropy + Weighted Dice (balanced for all 6 classes)  
**Dataset:** Low-light leather defects (.npy, 256×256 RGB)  
**Classes:** background(0), color(1), cut(2), fold(3), glue(4), poke(5)

---
### Why this configuration?
- ResNet50 encoder solves the `cut` and `poke` detection problem (pre-trained edge/texture features)
- Focal CE (γ=2) naturally focuses on hard/rare pixels without over-predicting defects
- Dice loss maximizes overlap and is class-balanced
- Square-root inverse frequency weights prevent background from being ignored

## 0. Colab Drive Mount

In [ ]:
import os
IS_COLAB = os.path.exists('/content')
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
else:
    print('Running locally.')

## 1. Imports & Configuration

In [ ]:
import os, warnings, json
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras import backend as K

print(f'TensorFlow : {tf.__version__}')
print(f'GPUs       : {tf.config.list_physical_devices("GPU")}')

# ── Paths ─────────────────────────────────────────────────────────────────────
if IS_COLAB:
    DATA_DIR    = Path('/content/drive/MyDrive/image/leather_project/processed_dataset')
    RESULTS_DIR = Path('/content/drive/MyDrive/image/leather_project/attn_unet_v5')
else:
    DATA_DIR    = Path(r'C:\Users\User\Downloads\lowlight\processed_dataset')
    RESULTS_DIR = Path(r'd:\7th sem\Image processing and CV\Leather_defect_detection\attn_unet_v5')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset ───────────────────────────────────────────────────────────────────
IMG_SIZE     = (256, 256)
IMG_CHANNELS = 3
DEFECT_TYPES = ['color', 'cut', 'fold', 'glue', 'poke']
CLASS_NAMES  = ['background'] + DEFECT_TYPES
NUM_CLASSES  = len(CLASS_NAMES)   # 6
CLASS_MAP    = {dt: i+1 for i, dt in enumerate(DEFECT_TYPES)}

CLASS_COLORS = np.array([
    [  0,   0,   0],  # background — black
    [220,  50,  50],  # color      — red
    [ 50, 210,  50],  # cut        — green
    [ 50,  80, 255],  # fold       — blue
    [240, 200,   0],  # glue       — yellow
    [210,  50, 210],  # poke       — magenta
], dtype=np.uint8)

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE    = 8
EPOCHS        = 60
WARMUP_EPOCHS = 5
LEARNING_RATE = 2e-4
MIN_LR        = 1e-6
SEED          = 42

# Loss weights
FOCAL_CE_WEIGHT = 0.6   # Weighted focal cross-entropy
DICE_WEIGHT     = 0.4   # Weighted dice loss
FOCAL_GAMMA     = 2.0   # Standard focal modulation
AUX_LOSS_WEIGHT = 0.4   # Deep supervision branch
LABEL_SMOOTHING = 0.05  # Prevents overconfident predictions

# Oversampling — moderate values to avoid data bias
OVERSAMPLE = {'color': 3, 'cut': 4, 'fold': 2, 'glue': 2, 'poke': 5, 'good': 1}

print(f'Classes    : {CLASS_NAMES}')
print(f'Results    : {RESULTS_DIR}')
print(f'Loss       : {FOCAL_CE_WEIGHT}×FocalCE + {DICE_WEIGHT}×Dice | γ={FOCAL_GAMMA}')

## 2. Data Discovery

In [ ]:
def parse_defect_type(filename):
    stem = Path(filename).stem
    for sfx in ['_ll0', '_ll1', '_ll2', '_ll3']:
        stem = stem.replace(sfx, '')
    parts = stem.rsplit('_', 1)
    return parts[0] if len(parts) == 2 else stem


def discover_samples(split):
    samples = []
    img_dir  = DATA_DIR / split / 'images'
    ll_dir   = DATA_DIR / split / 'lowlight'
    mask_dir = DATA_DIR / split / 'masks'

    for p in sorted(img_dir.glob('*.npy')):
        mp = mask_dir / p.name
        if not mp.exists(): continue
        dt = parse_defect_type(p.name)
        samples.append((str(p), str(mp), CLASS_MAP.get(dt, 0), dt))

    if ll_dir.exists():
        for p in sorted(ll_dir.glob('*.npy')):
            stem = p.stem
            for sfx in ['_ll0', '_ll1', '_ll2', '_ll3']:
                stem = stem.replace(sfx, '')
            mp = mask_dir / f'{stem}.npy'
            if not mp.exists(): continue
            dt = parse_defect_type(p.name)
            samples.append((str(p), str(mp), CLASS_MAP.get(dt, 0), dt))
    return samples


train_samples = discover_samples('train')
val_samples   = discover_samples('val')
test_samples  = discover_samples('test')

for name, sl in [('Train', train_samples), ('Val', val_samples), ('Test', test_samples)]:
    ll = sum(1 for s in sl if '_ll' in s[0])
    c  = Counter(s[3] for s in sl)
    print(f'{name:5s}: {len(sl)} ({ll} low-light) | {dict(c)}')

## 3. CLAHE + Data Loading + Augmentation

In [ ]:
CLAHE = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))


def apply_clahe(img_f):
    u8  = (np.clip(img_f, 0, 1) * 255).astype(np.uint8)
    lab = cv2.cvtColor(u8, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = CLAHE.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32) / 255.0


def load_sample(img_path, mask_path, class_id, defect_type=None):
    img  = apply_clahe(np.load(img_path).astype(np.float32))
    mask = np.load(mask_path).astype(np.float32)
    mc   = np.zeros(IMG_SIZE, dtype=np.int32)
    mc[mask > 0.5] = class_id
    return img, mc


def augment(image, mask):
    """Safe augmentation — conservative scale to preserve tiny defects."""
    # Flips and rotations
    if np.random.rand() < 0.5:
        image = np.fliplr(image).copy();  mask = np.fliplr(mask).copy()
    if np.random.rand() < 0.5:
        image = np.flipud(image).copy();  mask = np.flipud(mask).copy()
    k = np.random.randint(0, 4)
    image = np.rot90(image, k).copy();    mask = np.rot90(mask, k).copy()

    # Conservative scale (0.85–1.15) — safe for tiny cut/poke defects
    if np.random.rand() < 0.4:
        sc   = np.random.uniform(0.85, 1.15)
        H, W = image.shape[:2]
        nH   = max(1, int(H * sc)); nW = max(1, int(W * sc))
        imgs = cv2.resize(image, (nW, nH), interpolation=cv2.INTER_LINEAR)
        msks = cv2.resize(mask.astype(np.float32), (nW, nH),
                          interpolation=cv2.INTER_NEAREST).astype(np.int32)
        if sc > 1.0:
            sh, sw = (nH-H)//2, (nW-W)//2
            image, mask = imgs[sh:sh+H, sw:sw+W], msks[sh:sh+H, sw:sw+W]
        else:
            ph, pw = (H-nH)//2, (W-nW)//2
            ni = np.zeros_like(image); nm = np.zeros((H,W), dtype=np.int32)
            ni[ph:ph+nH, pw:pw+nW] = imgs; nm[ph:ph+nH, pw:pw+nW] = msks
            image, mask = ni, nm

    # Brightness
    if np.random.rand() < 0.5:
        image = np.clip(image * np.random.uniform(0.7, 1.3), 0, 1)
    # Contrast
    if np.random.rand() < 0.4:
        f = np.random.uniform(0.7, 1.3)
        m = image.mean(axis=(0, 1), keepdims=True)
        image = np.clip((image - m)*f + m, 0, 1)
    # HSV jitter
    if np.random.rand() < 0.5:
        u8  = (image * 255).astype(np.uint8)
        hsv = cv2.cvtColor(u8, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:,:,0] = (hsv[:,:,0] + np.random.uniform(-12, 12)) % 180
        hsv[:,:,1] = np.clip(hsv[:,:,1] * np.random.uniform(0.75, 1.25), 0, 255)
        image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)/255.0
    # Low-light gamma simulation
    if np.random.rand() < 0.4:
        image = np.clip(np.power(image, np.random.uniform(1.0, 1.8)), 0, 1)
    # Gaussian noise
    if np.random.rand() < 0.35:
        image = np.clip(image + np.random.normal(0, 0.01, image.shape).astype(np.float32), 0, 1)
    # Blur
    if np.random.rand() < 0.25:
        ks = np.random.choice([3, 5])
        image = cv2.GaussianBlur(image, (ks, ks), 0)
    return image.astype(np.float32), mask.astype(np.int32)


def make_dataset(samples, aug=False, repeat=True):
    imgs  = [s[0] for s in samples]
    masks = [s[1] for s in samples]
    cids  = [s[2] for s in samples]

    def gen():
        idxs = list(range(len(imgs)))
        if aug: np.random.shuffle(idxs)
        for i in idxs:
            img, msk = load_sample(imgs[i], masks[i], cids[i])
            if aug: img, msk = augment(img, msk)
            yield img, msk

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec((256, 256, 3), tf.float32),
            tf.TensorSpec((256, 256),    tf.int32),
        )
    )
    if repeat: ds = ds.repeat()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


print('Data pipeline ready.')

## 4. Class Weights & Oversampling

Using **square-root inverse frequency** weights to prevent extreme ratios that destabilize training.

In [ ]:
def compute_sqrt_class_weights(samples):
    """
    Square-root inverse frequency weights.
    Rationale: pure inverse freq gives background weight ~0.001 and poke ~1000,
    which causes the model to ignore background. sqrt dampens this extremity
    while still boosting rare defect classes.
    """
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for ip, mp, cid, _ in samples:
        m = np.load(mp)
        counts[0]   += int(np.sum(m <= 0.5))
        if cid > 0: counts[cid] += int(np.sum(m > 0.5))

    total = counts.sum()
    freq  = counts / total

    # Square-root inverse frequency: w_i = sqrt(1 / freq_i)
    w = np.zeros(NUM_CLASSES, dtype=np.float32)
    for i in range(NUM_CLASSES):
        if counts[i] > 0:
            w[i] = np.sqrt(1.0 / freq[i])

    # Normalize so that mean(defect_weights) = 1.0, and background is relative
    defect_mean = np.mean(w[1:])
    w /= defect_mean

    print('Class weights (sqrt-inv-freq, normalized to defect mean=1):')
    for i in range(NUM_CLASSES):
        pct = freq[i]*100 if total else 0
        print(f'  {CLASS_NAMES[i]:<12} {counts[i]:>10,} px  ({pct:.4f}%)  weight={w[i]:.4f}')
    return w


unique_train = [s for s in train_samples if '_ll' not in s[0]]
CLASS_WEIGHTS = compute_sqrt_class_weights(unique_train)
TF_CW = tf.constant(CLASS_WEIGHTS, dtype=tf.float32)


def do_oversample(samples):
    out = []
    for s in samples:
        out.extend([s] * OVERSAMPLE.get(s[3], 1))
    np.random.seed(SEED)
    np.random.shuffle(out)
    return out


train_os = do_oversample(train_samples)
print(f'\nTrain: {len(train_samples)} → oversampled: {len(train_os)}')

train_ds = make_dataset(train_os,    aug=True,  repeat=True)
val_ds   = make_dataset(val_samples, aug=False, repeat=True)

STEPS = max(len(train_os) * 2 // BATCH_SIZE, 1)
VSTEP = max(len(val_samples)  // BATCH_SIZE, 1)
print(f'Steps/epoch: {STEPS} | Val steps: {VSTEP}')

## 5. Attention UNet with ResNet50 Encoder

In [ ]:
def squeeze_excite(x, ratio=8):
    f  = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(f//ratio, 4), activation='relu')(se)
    se = layers.Dense(f, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, f))(se)
    return layers.Multiply()([x, se])


def conv_block(x, filters, drop=0.0):
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = squeeze_excite(x)
    if drop > 0: x = layers.SpatialDropout2D(drop)(x)
    return x


def attn_decoder_block(up_in, skip, filters, drop=0.0):
    """Upsample → Attention gate → Concatenate → conv_block"""
    f_att = max(filters // 2, 4)
    theta  = layers.Conv2D(f_att, 1, padding='same', use_bias=False)(skip)
    phi    = layers.Conv2D(f_att, 1, padding='same', use_bias=False)(up_in)
    att    = layers.Activation('relu')(layers.Add()([theta, phi]))
    att    = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(att)
    gated  = layers.Multiply()([skip, att])
    x      = layers.Concatenate()([up_in, gated])
    return conv_block(x, filters, drop=drop)


def build_model(num_classes=NUM_CLASSES):
    inp = layers.Input((256, 256, 3), name='image_input')

    # ── Preprocess for ResNet50 ───────────────────────────────────────────────
    x_pre = layers.Lambda(
        lambda t: tf.keras.applications.resnet50.preprocess_input(t * 255.0),
        name='preprocess'
    )(inp)

    # ── Encoder: ResNet50 (ImageNet pretrained, top-40 layers trainable) ──────
    backbone = tf.keras.applications.ResNet50(
        include_top=False, weights='imagenet', input_tensor=x_pre
    )
    for layer in backbone.layers[:-40]:
        layer.trainable = False
    for layer in backbone.layers[-40:]:
        layer.trainable = True

    # Skip connections
    s1 = inp                                                          # 256×256,    3
    s2 = backbone.get_layer('conv1_relu').output                      # 128×128,   64
    s3 = backbone.get_layer('conv2_block3_out').output                #  64×64,   256
    s4 = backbone.get_layer('conv3_block4_out').output                #  32×32,   512
    bn = backbone.get_layer('conv4_block6_out').output                #  16×16,  1024

    # Bottleneck projection
    b  = conv_block(bn, 512, drop=0.3)                                #  16×16,   512

    # ── Decoder ───────────────────────────────────────────────────────────────
    u4 = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(b) #  32×32
    d4 = attn_decoder_block(u4, s4, 256, drop=0.2)

    u3 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(d4) # 64×64
    d3 = attn_decoder_block(u3, s3, 128, drop=0.15)

    # ── Deep supervision (64×64 → 256×256) ───────────────────────────────────
    aux_up = layers.UpSampling2D(size=(4, 4), interpolation='bilinear')(d3)
    aux    = layers.Conv2D(num_classes, 1, activation='softmax', name='aux_output')(aux_up)

    u2 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(d3)  # 128×128
    d2 = attn_decoder_block(u2, s2, 64, drop=0.10)

    u1 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(d2)  # 256×256
    d1 = attn_decoder_block(u1, s1, 32, drop=0.05)

    out = layers.Conv2D(num_classes, 1, activation='softmax', name='main_output')(d1)

    return Model(inp, [out, aux], name='AttnUNet_ResNet50_v5')


print('Architecture defined.')

## 6. Loss Functions

| Component | Weight | Purpose |
|-----------|--------|---------|
| **Weighted Focal CE** | 0.6 | Focuses on hard pixels; won't over-predict defects |
| **Weighted Dice** | 0.4 | Maximises overlap for every class |

Both are weighted by **sqrt-inverse-frequency class weights** (see Cell 4).

In [ ]:
SMOOTH = 1e-6


def weighted_focal_ce_loss(y_true, y_pred):
    """
    Weighted Focal Cross-Entropy.
    - Per-pixel class weights from sqrt-inv-freq
    - gamma=2: focuses on hard/rare pixels
    - label smoothing prevents overconfidence
    """
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.clip_by_value(tf.reshape(y_pred, [-1, NUM_CLASSES]), 1e-7, 1.0 - 1e-7)

    # Label smoothing
    oh  = tf.one_hot(y_t, NUM_CLASSES)
    oh_smooth = oh * (1 - LABEL_SMOOTHING) + LABEL_SMOOTHING / NUM_CLASSES

    # Cross-entropy
    ce  = -tf.reduce_sum(oh_smooth * tf.math.log(y_p), axis=-1)   # (N,)

    # Focal modulation: (1 - p_t)^gamma
    pt  = tf.reduce_sum(oh * y_p, axis=-1)
    focal_mod = tf.pow(1.0 - pt, FOCAL_GAMMA)

    # Per-pixel class weight
    pw  = tf.gather(TF_CW, y_t)

    return tf.reduce_mean(pw * focal_mod * ce)


def weighted_dice_loss(y_true, y_pred):
    """
    Per-class weighted Dice loss.
    Weights are sqrt-inv-freq class weights.
    """
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.reshape(y_pred, [-1, NUM_CLASSES])
    oh  = tf.one_hot(y_t, NUM_CLASSES)

    inter = tf.reduce_sum(oh * y_p, axis=0)                  # (C,)
    union = tf.reduce_sum(oh + y_p, axis=0)                  # (C,)
    dice  = (2.0 * inter + SMOOTH) / (union + SMOOTH)        # (C,)

    # Normalize class weights
    dw = TF_CW / tf.reduce_sum(TF_CW)
    return 1.0 - tf.reduce_sum(dice * dw)


def combined_loss(y_true, y_pred):
    return (FOCAL_CE_WEIGHT * weighted_focal_ce_loss(y_true, y_pred) +
            DICE_WEIGHT     * weighted_dice_loss(y_true, y_pred))


# ── Metrics ───────────────────────────────────────────────────────────────────
def mean_iou_metric(y_true, y_pred):
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), -1), tf.int32)
    cm  = tf.cast(tf.math.confusion_matrix(y_t, y_p, NUM_CLASSES), tf.float32)
    d   = tf.linalg.diag_part(cm)
    den = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0) - d
    iou = tf.where(den > 0, d/den, tf.zeros_like(d))
    return tf.math.divide_no_nan(tf.reduce_sum(iou), tf.cast(tf.reduce_sum(tf.cast(den>0, tf.int32)), tf.float32))


def mean_dice_metric(y_true, y_pred):
    y_t = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
    y_p = tf.cast(tf.argmax(tf.reshape(y_pred, [-1, NUM_CLASSES]), -1), tf.int32)
    cm  = tf.cast(tf.math.confusion_matrix(y_t, y_p, NUM_CLASSES), tf.float32)
    d   = tf.linalg.diag_part(cm)
    den = tf.reduce_sum(cm, 1) + tf.reduce_sum(cm, 0)
    dc  = tf.where(den > 0, 2*d/den, tf.zeros_like(d))
    return tf.math.divide_no_nan(tf.reduce_sum(dc), tf.cast(tf.reduce_sum(tf.cast(den>0, tf.int32)), tf.float32))


print('Loss functions defined.')

## 7. Callbacks

In [ ]:
class CosineWarmupLR(callbacks.Callback):
    def __init__(self, max_lr, wu, total, min_lr=1e-6):
        super().__init__()
        self.max_lr, self.wu, self.total, self.min_lr = max_lr, wu, total, min_lr

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.wu:
            lr = self.max_lr * (epoch+1) / max(self.wu, 1)
        else:
            p  = (epoch - self.wu) / max(self.total - self.wu, 1)
            lr = self.min_lr + 0.5*(self.max_lr - self.min_lr)*(1 + np.cos(np.pi*p))
        try:    self.model.optimizer.learning_rate.assign(lr)
        except: K.set_value(self.model.optimizer.learning_rate, lr)


print('Callbacks ready.')

## 8. Build & Compile

In [ ]:
tf.keras.backend.clear_session()
np.random.seed(SEED); tf.random.set_seed(SEED)

model = build_model(num_classes=NUM_CLASSES)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss={'main_output': combined_loss, 'aux_output': combined_loss},
    loss_weights={'main_output': 1.0, 'aux_output': AUX_LOSS_WEIGHT},
    metrics={'main_output': [mean_iou_metric, mean_dice_metric]},
)

trainable = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
frozen    = sum(int(np.prod(v.shape)) for v in model.non_trainable_variables)
print(f'Model      : {model.name}')
print(f'Total      : {model.count_params():,}')
print(f'Trainable  : {trainable:,}  |  Frozen: {frozen:,}')
model.summary(line_length=110)

## 9. Train

In [ ]:
MODEL_PATH = str(RESULTS_DIR / 'best_model_v5.keras')

def wrap_ds(ds):
    return ds.map(
        lambda x, y: (x, {'main_output': y, 'aux_output': y}),
        num_parallel_calls=tf.data.AUTOTUNE
    )

MONITOR = 'val_main_output_mean_iou_metric'

cb_list = [
    callbacks.ModelCheckpoint(MODEL_PATH, monitor=MONITOR, mode='max',
                              save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor=MONITOR, mode='max',
                            patience=20, restore_best_weights=True, verbose=1),
    CosineWarmupLR(LEARNING_RATE, WARMUP_EPOCHS, EPOCHS, MIN_LR),
    callbacks.CSVLogger(str(RESULTS_DIR / 'training_log.csv')),
]

print('='*70)
print(f'Model   : {model.name}')
print(f'Epochs  : {EPOCHS} | Batch: {BATCH_SIZE} | Steps: {STEPS}')
print(f'Loss    : {FOCAL_CE_WEIGHT}×FocalCE(γ={FOCAL_GAMMA}) + {DICE_WEIGHT}×Dice')
print(f'Weights : sqrt-inverse-frequency (damped)')
print(f'LR      : {LEARNING_RATE} → {MIN_LR} cosine + {WARMUP_EPOCHS} warmup')
print('='*70)

history = model.fit(
    wrap_ds(train_ds),
    validation_data=wrap_ds(val_ds),
    epochs=EPOCHS,
    steps_per_epoch=STEPS,
    validation_steps=VSTEP,
    callbacks=cb_list,
    verbose=1,
)

print('Training complete!')

## 10. Training History Plots

In [ ]:
def plot_history(history, save_path=None):
    h = history.history

    def find(keys):
        for k in keys:
            if k in h: return h[k]
        return []

    tl  = find(['loss'])
    vl  = find(['val_loss'])
    ti  = find(['main_output_mean_iou_metric',  'mean_iou_metric'])
    vi  = find(['val_main_output_mean_iou_metric', 'val_mean_iou_metric'])
    td  = find(['main_output_mean_dice_metric', 'mean_dice_metric'])
    vd  = find(['val_main_output_mean_dice_metric', 'val_mean_dice_metric'])
    ep  = range(1, len(tl)+1) if tl else []

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Training History', fontsize=15, fontweight='bold')

    for ax, title, tr, va in [
        (axes[0], 'Combined Loss', tl, vl),
        (axes[1], 'Mean IoU',     ti, vi),
        (axes[2], 'Mean Dice',    td, vd),
    ]:
        if tr: ax.plot(ep, tr, 'royalblue',  lw=2, label='Train')
        if va: ax.plot(ep, va, 'orangered',  lw=2, linestyle='--', label='Val')
        ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(alpha=0.3)

    # Mark best epoch
    if vi:
        best = int(np.argmax(vi))
        for ax in axes:
            ax.axvline(x=best+1, color='green', linestyle=':', alpha=0.7, label=f'Best (ep {best+1})')

    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Best val IoU: {max(vi):.4f} at epoch {np.argmax(vi)+1}' if vi else '')


plot_history(history, save_path=str(RESULTS_DIR / 'training_history.png'))

## 11. Load Best Checkpoint

In [ ]:
print('Loading best model checkpoint...')
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        'combined_loss'          : combined_loss,
        'weighted_focal_ce_loss' : weighted_focal_ce_loss,
        'weighted_dice_loss'     : weighted_dice_loss,
        'mean_iou_metric'        : mean_iou_metric,
        'mean_dice_metric'       : mean_dice_metric,
    }
)
print(f'Loaded: {MODEL_PATH}')

## 12. Full Quantitative Evaluation

In [ ]:
def predict_one(model, sample):
    img, gt = load_sample(sample[0], sample[1], sample[2])
    preds   = model.predict(img[np.newaxis], verbose=0)
    pred_map = preds[0][0] if isinstance(preds, list) else preds[0]
    pm      = np.argmax(pred_map, axis=-1)
    return img, gt, pm


def evaluate_set(model, samples, label=''):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for s in samples:
        _, gt, pm = predict_one(model, s)
        for tc in range(NUM_CLASSES):
            for pc in range(NUM_CLASSES):
                cm[tc, pc] += int(np.sum((gt == tc) & (pm == pc)))

    print(f'\n{"="*64}')
    print(f'  EVALUATION — {label}')
    print('='*64)
    print(f'  {"Class":<14} {"IoU":>8} {"Dice":>8} {"Precision":>10} {"Recall":>8}')
    print('  ' + '-'*52)
    ious, dices = [], []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        iou  = tp / (tp+fp+fn)    if (tp+fp+fn) else 0.0
        dice = 2*tp/(2*tp+fp+fn)  if (2*tp+fp+fn) else 0.0
        prec = tp / (tp+fp)       if (tp+fp) else 0.0
        rec  = tp / (tp+fn)       if (tp+fn) else 0.0
        ious.append(iou); dices.append(dice)
        print(f'  {CLASS_NAMES[c]:<14} {iou:>8.4f} {dice:>8.4f} {prec:>10.4f} {rec:>8.4f}')
    print('  ' + '-'*52)
    print(f'  {"Mean (all)":<14} {np.mean(ious):>8.4f} {np.mean(dices):>8.4f}')
    print(f'  {"Mean (defect)":<14} {np.mean(ious[1:]):>8.4f} {np.mean(dices[1:]):>8.4f}')
    return ious, dices, cm


val_ious,  val_dices,  val_cm  = evaluate_set(model, val_samples,  'Validation Set')
test_ious, test_dices, test_cm = evaluate_set(model, test_samples, 'Test Set')

val_normal = [s for s in val_samples if '_ll' not in s[0]]
val_ll     = [s for s in val_samples if '_ll'     in s[0]]
if val_normal: evaluate_set(model, val_normal, 'Val — Normal Images')
if val_ll:     evaluate_set(model, val_ll,     'Val — Low-Light Images')

## 13. Per-Class IoU & Dice Bar Charts

In [ ]:
def plot_metrics_bar(val_ious, test_ious, val_dices, test_dices, save_path=None):
    x  = np.arange(NUM_CLASSES)
    w  = 0.35
    colors_val  = [CLASS_COLORS[i]/255.0 for i in range(NUM_CLASSES)]
    colors_test = [np.clip(CLASS_COLORS[i]/255.0 * 0.65, 0, 1) for i in range(NUM_CLASSES)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Per-Class Segmentation Metrics', fontsize=14, fontweight='bold')

    for ax, vm, tm, ylabel in [
        (axes[0], val_ious,  test_ious,  'IoU'),
        (axes[1], val_dices, test_dices, 'Dice Coefficient'),
    ]:
        b1 = ax.bar(x-w/2, vm, w, color=colors_val,  label='Validation', edgecolor='k', lw=0.5)
        b2 = ax.bar(x+w/2, tm, w, color=colors_test, label='Test',       edgecolor='k', lw=0.5)
        ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=11); ax.set_ylim(0, 1.1)
        ax.set_title(ylabel, fontweight='bold'); ax.legend(); ax.grid(axis='y', alpha=0.3)
        ax.axhline(0.5, color='gray', linestyle=':', lw=1, alpha=0.7)
        for bar in list(b1)+list(b2):
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.012, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=7.5, fontweight='bold')

    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


plot_metrics_bar(val_ious, test_ious, val_dices, test_dices,
                 save_path=str(RESULTS_DIR / 'per_class_metrics.png'))

## 14. Confusion Matrix

In [ ]:
def plot_confusion_matrix(cm, title='Confusion Matrix', save_path=None):
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm  = np.where(row_sums > 0, cm.astype(np.float32) / row_sums, 0)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            clr = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center', color=clr, fontsize=9)
    ax.set_xlabel('Predicted Class', fontsize=11)
    ax.set_ylabel('True Class', fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=13)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


plot_confusion_matrix(val_cm,  title='Confusion Matrix — Validation Set',
                      save_path=str(RESULTS_DIR / 'cm_val.png'))
plot_confusion_matrix(test_cm, title='Confusion Matrix — Test Set',
                      save_path=str(RESULTS_DIR / 'cm_test.png'))

## 15. Prediction Visualizations — All Defect Classes

For each defect type: **Input | Ground Truth | Prediction | Overlay**

In [ ]:
def colorize(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for c in range(NUM_CLASSES): rgb[mask == c] = CLASS_COLORS[c]
    return rgb


def overlay_prediction(img_f, pred_mask, alpha=0.55):
    base = (np.clip(img_f, 0, 1) * 255).astype(np.uint8)
    col  = colorize(pred_mask)
    fg   = pred_mask > 0
    out  = base.copy()
    if fg.any():
        out[fg] = (alpha * col[fg].astype(np.float32) +
                   (1-alpha) * base[fg].astype(np.float32)).astype(np.uint8)
    return out


def visualize_predictions(model, samples, title='Predictions',
                          n_per_class=2, save_path=None):
    selected = []
    for dt in DEFECT_TYPES:
        cls_samps = [s for s in samples if s[3] == dt]
        selected.extend(cls_samps[:n_per_class])

    n = len(selected)
    if n == 0:
        print('No samples found.'); return

    fig, axes = plt.subplots(n, 4, figsize=(22, 5.2*n))
    if n == 1: axes = axes[np.newaxis, :]

    col_labels = ['Input Image', 'Ground Truth', 'Prediction', 'Overlay']
    for col, lbl in enumerate(col_labels):
        axes[0, col].set_title(lbl, fontsize=13, fontweight='bold', pad=8)

    for row, s in enumerate(selected):
        img, gt, pm = predict_one(model, s)

        # Per-image class IoU
        tp = np.sum((gt == s[2]) & (pm == s[2]))
        fp = np.sum((gt != s[2]) & (pm == s[2]))
        fn = np.sum((gt == s[2]) & (pm != s[2]))
        iou = tp / (tp+fp+fn) if (tp+fp+fn) else 0.0

        is_ll = '_ll' in s[0]
        tag   = f"{s[3].upper()}  {'[Low-Light]' if is_ll else '[Normal]'}"

        axes[row, 0].imshow(img)
        axes[row, 0].set_ylabel(tag, fontsize=10, rotation=90, va='center', labelpad=8)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(colorize(gt))
        axes[row, 1].axis('off')

        axes[row, 2].imshow(colorize(pm))
        axes[row, 2].set_title(f'IoU = {iou:.3f}', fontsize=9, color='green' if iou > 0.5 else 'red')
        axes[row, 2].axis('off')

        axes[row, 3].imshow(overlay_prediction(img, pm))
        axes[row, 3].axis('off')

    # Class legend
    patches = [mpatches.Patch(color=CLASS_COLORS[i]/255.0, label=CLASS_NAMES[i])
               for i in range(NUM_CLASSES)]
    fig.legend(handles=patches, loc='lower center', ncol=NUM_CLASSES,
               fontsize=10, framealpha=0.95, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


# Validation: 2 samples per defect class
visualize_predictions(model, val_samples,
                      title='Validation Predictions — All Defect Classes',
                      n_per_class=2,
                      save_path=str(RESULTS_DIR / 'val_predictions.png'))

# Test: 2 samples per defect class
visualize_predictions(model, test_samples,
                      title='Test Set Predictions — All Defect Classes',
                      n_per_class=2,
                      save_path=str(RESULTS_DIR / 'test_predictions.png'))

## 16. Failure Case Analysis

Automatically finds the **worst-performing samples** per defect class.

In [ ]:
def get_sample_iou(model, sample):
    _, gt, pm = predict_one(model, sample)
    cid = sample[2]
    tp  = np.sum((gt == cid) & (pm == cid))
    fp  = np.sum((gt != cid) & (pm == cid))
    fn  = np.sum((gt == cid) & (pm != cid))
    return tp / (tp+fp+fn) if (tp+fp+fn) else 0.0


def find_worst_k(model, samples, k=2):
    per_class = {dt: [] for dt in DEFECT_TYPES}
    for s in samples:
        if s[3] not in DEFECT_TYPES: continue
        iou = get_sample_iou(model, s)
        per_class[s[3]].append((iou, s))
    worst = []
    for dt, scores in per_class.items():
        scores.sort(key=lambda x: x[0])
        worst.extend([sc[1] for sc in scores[:k]])
    return worst


worst = find_worst_k(model, val_samples + test_samples, k=2)
visualize_predictions(model, worst,
                      title='Failure Cases — Lowest IoU Samples per Class',
                      n_per_class=10,
                      save_path=str(RESULTS_DIR / 'failure_cases.png'))

## 17. Summary & File List

In [ ]:
print('\n' + '='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print(f'  Model      : {model.name}')
print(f'  Parameters : {model.count_params():,}')
print(f'  Loss       : {FOCAL_CE_WEIGHT}×FocalCE + {DICE_WEIGHT}×Dice')
print(f'  Val  — Mean IoU (defect): {np.mean(val_ious[1:]):.4f}  Dice: {np.mean(val_dices[1:]):.4f}')
print(f'  Test — Mean IoU (defect): {np.mean(test_ious[1:]):.4f}  Dice: {np.mean(test_dices[1:]):.4f}')
print('='*60)

print('\nSaved files:')
for f in sorted(RESULTS_DIR.glob('*')):
    kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {kb:>8.1f} KB')

# Download model checkpoint (Colab)
if IS_COLAB and os.path.exists(MODEL_PATH):
    from google.colab import files
    files.download(MODEL_PATH)
    print('\nModel download started.')